# MLOps Training 2026/2027 — Task 2

## Notebook 3 — Train / Validation / Test Split

This notebook reads the labeled artifact, chooses a chronological split to simulate future prediction, checks date ranges and label balance, and saves train, validation, and test files.

**Input:** `artifacts/02_labeled_table.parquet`

**Outputs:** `03_train.parquet`, `03_validation.parquet`, `03_test.parquet`


## 1. Load and order the labeled data

A time-based split is selected because the business use case is to predict future delivery performance using historical orders. This prevents future orders from entering the training period.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\HP\Desktop\MLOps")
TASK2_ROOT = PROJECT_ROOT / "Task2"
ARTIFACTS_DIR = TASK2_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

input_path = ARTIFACTS_DIR / "tables" / "02_labeled_table.parquet"
df = pd.read_parquet(input_path)
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")
df = df.dropna(subset=["order_purchase_timestamp", "is_late"]).sort_values("order_purchase_timestamp").reset_index(drop=True)
assert df["order_id"].is_unique
print(df["order_purchase_timestamp"].min(), "to", df["order_purchase_timestamp"].max())
print(f"Rows available: {len(df):,}")


2016-09-15 12:16:38 to 2018-08-29 15:00:37
Rows available: 96,470


## 2. Create chronological train, validation, and test splits

The proportions are 70% train, 15% validation, and 15% test. No stratification is used because time order is the priority; label balance is checked afterward.


In [2]:
n = len(df)
train_end = int(n * 0.70)
validation_end = int(n * 0.85)
train = df.iloc[:train_end].copy()
validation = df.iloc[train_end:validation_end].copy()
test = df.iloc[validation_end:].copy()

assert set(train.order_id).isdisjoint(validation.order_id)
assert set(train.order_id).isdisjoint(test.order_id)
assert set(validation.order_id).isdisjoint(test.order_id)

def split_summary(name, part):
    return {"split": name, "rows": len(part), "start": part.order_purchase_timestamp.min(), "end": part.order_purchase_timestamp.max(), "late_count": int(part.is_late.sum()), "late_percentage": round(100 * part.is_late.mean(), 2)}

split_stats = pd.DataFrame([split_summary("train", train), split_summary("validation", validation), split_summary("test", test)])
display(split_stats)


,split,rows,start,end,late_count,late_percentage
0,train,67529,2016-09-15 12:16:38,2018-04-15 20:12:35,6096,9.03
1,validation,14470,2018-04-15 20:17:11,2018-06-21 08:29:29,773,5.34
2,test,14471,2018-06-21 08:41:07,2018-08-29 15:00:37,957,6.61


## 3. Save split artifacts


In [3]:
paths = {"train": ARTIFACTS_DIR / "tables" / "03_train.parquet", "validation": ARTIFACTS_DIR / "tables" / "03_validation.parquet", "test": ARTIFACTS_DIR / "tables" / "03_test.parquet"}
for name, part in [("train", train), ("validation", validation), ("test", test)]:
    part.to_parquet(paths[name], index=False)

metadata = {"strategy": "chronological", "proportions": {"train": 0.70, "validation": 0.15, "test": 0.15}, "reason": "Simulate future prediction and avoid placing later orders in training", "splits": split_stats.astype(str).to_dict(orient="records")}
(ARTIFACTS_DIR / "reports" / "03_split_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved train, validation, and test artifacts.")


Saved train, validation, and test artifacts.


In [4]:
for path in paths.values():
    assert path.exists() and path.stat().st_size > 0
print("ALL NOTEBOOK 3 CHECKS PASSED")


ALL NOTEBOOK 3 CHECKS PASSED
